# Gemma-3-1B Activation Precomputation (GPU T4)

Этот ноутбук предназначен для предварительного вычисления активаций (hidden states) перед слоем Titans (слой 23) модели Gemma-3-1B. Это позволяет значительно ускорить обучение TitansBlock, так как графу JAX не нужно просчитывать первые 23 слоя Gemma при каждом шаге.

**Особенности:**
- Оптимизировано для GPU T4 (Colab Free/Pro).
- Автоматическая выгрузка шардов в HuggingFace Hub.
- Поддержка докачки (resume) при обрыве сессии.

In [ ]:
# 1. Установка зависимостей
!pip install -q --upgrade "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q git+https://github.com/google-deepmind/gemma.git
!pip install -q flax==0.12.5 optax==0.2.6 typeguard==4.4.1 datasets ml_dtypes huggingface_hub orbax-checkpoint

In [ ]:
# 2. Аутентификация HuggingFace
from huggingface_hub import login
import os

# Вставьте свой токен или используйте секреты Colab
hf_token = "YOUR_HF_TOKEN_HERE"
if hf_token == "YOUR_HF_TOKEN_HERE":
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
    except:
        print("Пожалуйста, укажите hf_token или добавьте его в секреты Colab под именем HF_TOKEN")

login(token=hf_token)

In [ ]:
# 3. Загрузка чекпоинта Gemma-3-1B
!mkdir -p gemma3_1b_ckpt
# !gsutil -m cp -r gs://gemma-data/checkpoints/gemma3-1b-it ./gemma3_1b_ckpt

In [ ]:
# 4. Конфигурация
config = {
    "gemma_ckpt": "./gemma3_1b_ckpt/gemma3-1b-it",
    "dataset_repo": "veriga/openwebtext-gemma3-tokenized-1024",
    "output_dir": "./activations_layer23",
    "target_layer": 23,
    "max_seq_len": 1024,
    "batch_size": 4,          # BS=4 безопасно для T4 (16GB)
    "dtype": "bfloat16",      # JAX на GPU умеет эмулировать bf16
    "save_dtype": "bfloat16",
    "max_examples": None,     # None = весь датасет
    "resume": True,
    "push_activations": True,
    "activations_repo": "veriga/openwebtext-gemma3-tokenized-1024-activations-layer23",
    "upload_batch": 64,       # Делать коммит в HF каждые 64 шарда
    "upload_workers": 2
}

os.makedirs(config["output_dir"], exist_ok=True)

In [ ]:
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import jax
import jax.numpy as jnp
import numpy as np
from gemma import gm
from datasets import load_dataset

def make_forward_to_layer(target_layer: int):
    model = gm.nn.Gemma3_1B()
    @jax.jit
    def forward_fn(params, tokens):
        output = model.apply({'params': params}, tokens, return_hidden_states=True)
        return output.hidden_states[target_layer]
    return forward_fn

def dataset_to_batches(ds, batch_size, max_seq_len, max_examples=None):
    total = len(ds)
    if max_examples: total = min(total, max_examples)
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = ds[start:end]
        raw_tokens = batch.get("tokens") or batch.get("input_ids")
        
        batch_tokens, batch_masks = [], []
        for tokens in raw_tokens:
            t_arr = np.array(tokens[:max_seq_len], dtype=np.int32)
            orig_len = len(t_arr)
            pad_len = max_seq_len - orig_len
            if pad_len > 0: t_arr = np.pad(t_arr, (0, pad_len))
            m_arr = np.zeros(max_seq_len, dtype=np.int32)
            m_arr[:orig_len] = 1
            batch_tokens.append(t_arr)
            batch_masks.append(m_arr)
        yield np.stack(batch_tokens), np.stack(batch_masks)

def upload_shard_batch(shard_paths, repo_id, token, max_retries=3):
    from huggingface_hub import CommitOperationAdd, create_commit
    ops = [CommitOperationAdd(path_in_repo=os.path.basename(p), path_or_fileobj=p) for p in shard_paths]
    for attempt in range(max_retries):
        try:
            create_commit(repo_id=repo_id, operations=ops, 
                          commit_message=f"Add {len(shard_paths)} shards",
                          token=token, repo_type="dataset")
            return True
        except Exception as e:
            print(f"Upload error: {e}")
            time.sleep(10 * (2**attempt))
    return False

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print("🔧 Загрузка параметров...")
params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
forward_fn = make_forward_to_layer(config["target_layer"])

print("📂 Загрузка датасета...")
ds = load_dataset(config["dataset_repo"], 
                  split="train",
                  cache_dir="/content/drive/Shareddrives/shared_veriga/jax_cache"
                  )

start_shard = 0
meta_path = os.path.join(config["output_dir"], "metadata.json")
if config["resume"] and os.path.exists(meta_path):
    with open(meta_path) as f: start_shard = json.load(f).get("next_shard", 0)
    print(f"📋 Возобновление с шарда {start_shard}")

executor = ThreadPoolExecutor(max_workers=config["upload_workers"])
pending_shards = []
t_start = time.time()
shard_idx = start_shard

save_dtype = jnp.bfloat16 if config["save_dtype"] == "bfloat16" else np.float32

batch_gen = dataset_to_batches(ds, config["batch_size"], config["max_seq_len"], config["max_examples"])
for _ in range(start_shard): next(batch_gen, None)

print(f"🚀 Начало обработки (GPU: {jax.devices()[0].device_kind})...")
for batch_tokens, batch_masks in batch_gen:
    hidden = forward_fn(params, jnp.array(batch_tokens))
    hidden_np = np.asarray(hidden) * batch_masks[:, :, None].astype(np.float32)
    
    paths = []
    for suffix, data in [("", hidden_np.astype(save_dtype)), ("_tokens", batch_tokens), ("_masks", batch_masks)]:
        path = os.path.join(config["output_dir"], f"shard_{shard_idx:06d}{suffix}.npy")
        np.save(path, data)
        paths.append(path)
    
    if config["push_activations"]:
        pending_shards.extend(paths)
        if len(pending_shards) >= config["upload_batch"] * 3:
            executor.submit(upload_shard_batch, pending_shards[:], config["activations_repo"], hf_token)
            pending_shards = []
            
    shard_idx += 1
    if shard_idx % 10 == 0:
        elapsed = time.time() - t_start
        print(f"  Shard {shard_idx} | {shard_idx*config['batch_size']/elapsed:.1f} ex/s | {elapsed:.1f}s")
        with open(meta_path, "w") as f: json.dump({"next_shard": shard_idx}, f)

print("✅ Обработка завершена!")